In [1]:
%matplotlib inline

# Check the ensemble simulation results for HBR
# and find the best parameter set for runoff and LAI
import os
import sys
sys.path.append(os.path.join(os.environ['HOME'], 'models', 'OLMT'))

import pickle
import model_ELM
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
from shared_read_HBR import *

file = open(os.path.join(os.environ['HOME'], 'models', 'OLMT', 'pklfiles',
                         '20250422_HBR_ICB20TRCNPRDCTCBC_6year_rmethod1erw.pkl'), 'rb')
model = pickle.load(file)

In [2]:
tlai_rmse = np.sqrt(np.sum(np.power(model.output['TLAI_ann'] - \
                                    model.obs['TLAI_ann'].reshape(-1,1), 2), axis = 0))
runoff_rmse = np.sqrt(np.sum(np.power(model.output['QRUNOFF_ann'] - \
                                      model.obs['QRUNOFF_ann'].reshape(-1,1), 2), axis = 0))

best_ind = np.argmin(tlai_rmse**2 + runoff_rmse**2)
print('ensemble = ', best_ind+1)

ensemble =  2927


In [3]:
data = np.vstack([tlai_rmse, runoff_rmse]).T

kde = gaussian_kde(data.T)
densities = kde(data.T)

# Sort the points by density for better coloring
idx = densities.argsort()
x, y = data[idx, 0], data[idx, 1]
densities = densities[idx]

# Plot
fig, ax = plt.subplots(figsize = (8, 6))
scatter = ax.scatter(x, y, c=densities, s=30, cmap='viridis')
ax.scatter(tlai_rmse[best_ind], runoff_rmse[best_ind], s=100, color = 'r', 
           facecolor = 'none', marker = 's')
plt.colorbar(scatter, ax = ax, label='Local Density')
ax.set_xlabel('RMSE TLAI')
ax.set_ylabel('RMSE Runoff')
ax.set_title('Scatter Plot Colored by Local Density')
plt.savefig(os.path.join(os.environ['HOME'], 'HBR_calibration.png'), dpi = 600.)